# Isolation Forest trên KPI `02e99bd4`

In [15]:
import sys, time
sys.path.append(r"C:\Projects\anomaly-detection\Modeling\Code")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score
from eval_protocol import time_split_per_kpi, evaluate_protocol, plot_pr_curve
from preprocess import preprocess_all

## 1. Đọc KPI

In [ ]:
KPI_PREFIX = "02e99bd4"
df = pd.read_csv(r"C:\Projects\anomaly-detection\Data\train.csv")
df.columns = ["timestamp", "value", "label", "kpi"]
kpi = [k for k in df.kpi.unique() if k.startswith(KPI_PREFIX)][0]
g = df[df.kpi == kpi].copy()
step = int(pd.Series(np.diff(np.sort(g.timestamp.values))).mode().iloc[0])

g = time_split_per_kpi(g, train_frac=0.6, val_frac=0.2)
p = preprocess_all(g, max_gap_points=5, norm_method="robust")
p = p.sort_values("timestamp").reset_index(drop=True)

print("KPI:", kpi, "| step:", step, "s")
for s in ["train", "val", "test"]:
    sub = p[p.split == s]
    print(f"  {s:5s}: {len(sub):6d} điểm | anomaly={int(sub.label.sum()):5d} ({sub.label.mean()*100:.2f}%)")

KPI: 02e99bd4f6cfb33f | step: 60 s
  train:  80169 điểm | anomaly= 8444 (10.53%)
  val  :  25718 điểm | anomaly=  724 (2.82%)
  test :  25908 điểm | anomaly= 1382 (5.33%)


## 2. Shingling + hàm train/score

In [17]:
v   = p.value_norm.values.astype(float)
lab = p.label.values.astype(int)
spl = p.split.values
_cache = {}

def make_shingles(s):
    if s in _cache:
        return _cache[s]
    W   = sliding_window_view(v, s)          # (N-s+1, s)
    cur = np.arange(s - 1, len(v))           # điểm hiện tại = cuối cửa sổ
    ok  = ~np.isnan(W).any(axis=1)           # loại cửa sổ chạm gap
    _cache[s] = (W[ok], lab[cur][ok], spl[cur][ok])
    return _cache[s]

def fit_score(S, max_samples, seed, clean):
    X, y, sp = make_shingles(S)
    tr = sp == "train"
    fit_mask = tr & (y == 0) if clean else tr
    clf = IsolationForest(n_estimators=100, max_samples=max_samples,
                          contamination="auto", random_state=seed)
    t0 = time.perf_counter(); clf.fit(X[fit_mask]);           t_fit = time.perf_counter() - t0
    t0 = time.perf_counter(); sc = -clf.decision_function(X); t_inf = (time.perf_counter() - t0) / len(X)
    return y, sp, sc, t_fit, t_inf

## 3. Tune `shingle_size` + `max_samples`

In [19]:
print(f"{'shingle':>7} {'max_samples':>11} | {'AP_val':>7}")
best = None
for S_ in [8, 16, 32, 48, 64]:
    for ms in [128, 256, 512]:
        y, sp, sc, _, _ = fit_score(S_, ms, 42, clean=False)
        va = sp == "val"
        ap = average_precision_score(y[va], sc[va])
        print(f"{S_:>7} {ms:>11} | {ap:>7.4f}")
        if best is None or ap > best[0]:
            best = (ap, S_, ms)
_, S, MS = best
print(f"=> shingle={S}, max_samples={MS} (AP_val={best[0]:.4f})")

shingle max_samples |  AP_val
      8         128 |  0.4128
      8         256 |  0.5523
      8         512 |  0.6135
     16         128 |  0.4801
     16         256 |  0.6409
     16         512 |  0.6909
     32         128 |  0.5727
     32         256 |  0.7211
     32         512 |  0.7738
     48         128 |  0.5927
     48         256 |  0.6846
     48         512 |  0.7768
     64         128 |  0.5225
     64         256 |  0.6318
     64         512 |  0.7262
=> shingle=48, max_samples=512 (AP_val=0.7768)


In [ ]:
def ap_val(clean):
    y, sp, sc, _, _ = fit_score(S, MS, 42, clean)
    return average_precision_score(y[sp == "val"], sc[sp == "val"])

apv_dirty, apv_clean = ap_val(False), ap_val(True)
CLEAN = apv_clean >= apv_dirty
print(f"AP_val  apv_dirty={apv_dirty:.4f} | apv_clean={apv_clean:.4f})

AP_val  apv_dirty=0.7768 | apv_clean=0.6786  -> dùng dirty data


## 5. Model cuối: multi-seed (mean ± std) + thời gian train/infer

In [22]:
seeds = [0, 1, 2, 3, 4]
rows, tf, ti = [], [], []
for sd in seeds:
    y, sp, sc, t_fit, t_inf = fit_score(S, MS, sd, clean=CLEAN)
    r = evaluate_protocol(y[sp == "val"], sc[sp == "val"],
                          y[sp == "test"], sc[sp == "test"], beta=1.0, step_s=step)
    rows.append((r["PW"]["fbeta"], r["PW"]["precision"], r["PW"]["recall"],
                 r["PA"]["fbeta"], r["AP_pw"]))
    tf.append(t_fit); ti.append(t_inf)

a = np.array(rows); mean, std = a.mean(0), a.std(0)
print(f"=== Isolation Forest | KPI {kpi[:8]} | shingle={S}, max_samples={MS}, {len(seeds)} seeds ===")
for i, nm in enumerate("PW_F1 PW_P PW_R PA_F1 AP".split()):
    print(f"  {nm:6s}: {mean[i]:.3f} ± {std[i]:.3f}")
print(f"  fit_time ~ {np.mean(tf)*1000:.0f} ms | infer/điểm ~ {np.mean(ti)*1e6:.2f} µs")
# TTD là xấp xỉ (shingle chạm gap bị loại làm timeline không hoàn toàn liên tục).

=== Isolation Forest | KPI 02e99bd4 | shingle=48, max_samples=512, 5 seeds ===
  PW_F1 : 0.673 ± 0.001
  PW_P  : 0.571 ± 0.008
  PW_R  : 0.820 ± 0.015
  PA_F1 : 0.850 ± 0.037
  AP    : 0.796 ± 0.005
  fit_time ~ 241 ms | infer/điểm ~ 5.94 µs
